In [36]:
pip install -q youtube-transcript-api langchain-community faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [37]:
%pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [38]:
import langchain_text_splitters

In [39]:
from youtube_transcript_api import YouTubeTranscriptApi , TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [63]:
pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [68]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2:latest")
response = llm.invoke("Hello")
print(response)
print(response.content if hasattr(response, "content") else response)

Hello! How can I assist you today?
Hello! How can I assist you today?


In [61]:
from langchain_community.vectorstores import FAISS 
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import PromptTemplate

INDEXING


In [42]:
video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()

    transcript = api.fetch(video_id, languages=['en'])

    # Convert transcript snippets into a single string
    transcript_text = " ".join(
        snippet.text for snippet in transcript.snippets
    )

    print("Transcript fetched successfully!")
    print(transcript_text[:20000])

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")  

Transcript fetched successfully!
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to intervie

Text Spliting


In [43]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
chunks = splitter.split_text(transcript_text)

In [44]:
len(chunks)

168

In [45]:
chunks[0]

"the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough

Embedding Generation and Storing in Vector Store 

In [46]:
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_texts(chunks, embeddings_model)

In [47]:
vector_store.index_to_docstore_id

{0: 'd3c7f127-72d5-4dd4-adeb-9725c4d0a3a2',
 1: 'c4312304-3597-4029-8787-35cf43b1c458',
 2: 'f23693a8-a0d3-47a1-94aa-cb9eb7a6349a',
 3: '4c36e359-28be-424c-9a0d-6b29e8a035cd',
 4: '4d4d8d02-5ecd-474b-8971-b24f2670df54',
 5: '4db65c12-be80-438e-84bf-aef1de986af0',
 6: '2c831c33-0e53-4623-98c6-ff5dd54868bb',
 7: 'dd565e54-6841-48f0-81ec-e6dedcb440af',
 8: '9c25aaa7-abde-4a12-a683-4bc7c67c9b2c',
 9: 'a24f674a-bf1e-4a44-976e-05a7848a7c40',
 10: '8b9cb473-2001-4b75-818f-6242914e7a64',
 11: '1db6e5b5-10bd-4b40-a89f-78d4b604167c',
 12: '198f7d5d-002c-43d9-adaf-1ff0d8bf68fd',
 13: '83d43dc3-bee7-4f97-baf1-ecab0fe31504',
 14: '395d5533-2e58-4c4e-b77a-0b4597e26041',
 15: '2485e1eb-3024-4304-ab16-cab6562a2762',
 16: '9b1a1efd-b23c-403b-aba9-31b63077f620',
 17: 'c460585c-7ab0-44c1-a928-1008c9bf0d96',
 18: '3dc23006-d7b2-484c-93db-6d651364ccfb',
 19: 'eb0e44a1-8dcc-4539-84b6-cbfe9b9fdc6e',
 20: 'a712644a-d537-4834-9d0d-ee219f6eadf3',
 21: 'e17cf78e-0c5b-498f-a8b3-35ca55183c31',
 22: '12997850-a952-

RETRIEVAL

In [48]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [49]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A57CD291D0>, search_kwargs={'k': 3})

In [50]:
retriever.invoke("What is deepmind?")

[Document(id='466805b8-df91-4450-add9-7f7666202c4a', metadata={}, page_content='i used to discuss um uh uh what were the sort of founding tenets of deep mind and it was very various things one was um algorithmic advances so deep learning you know jeff hinton and cohen just had just sort of invented that in academia but no one in industry knew about it uh we love reinforcement learning we thought that could be scaled up but also understanding about the human brain had advanced um quite a lot uh in the decade prior with fmri machines and other things so we could get some good hints about architectures and algorithms and and sort of um representations maybe that the brain uses so as at a systems level not at a implementation level um and then the other big things were compute and gpus right so we could see a compute was going to be really useful and it got to a place where it became commoditized mostly through the games industry and and that could be taken advantage of and then the final 

Auggumentation

In [69]:
llm = OllamaLLM(model="llama3.2")

In [70]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="You are a helpful assistant. Answer ONLY from the provided transcript context. If the context is insufficient , just say you don't know.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

In [78]:
question = "is the topic of Nuclear fusion discussed in the video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [79]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [80]:
context_text

"that i would like to i think could be very transformative if we helped accelerate and uh really interesting problems scientific challenges in of themselves this is energy so energy yes exactly so energy and climate so we talked about disease and biology as being one of the biggest places i think ai can help with i think energy and climate uh is another one so maybe they would be my top two um and fusion is one one area i think ai can help with now fusion has many challenges mostly physics material science and engineering challenges as well to build these massive fusion reactors and contain the plasma and what we try to do whenever we go into a new field to apply our systems is we look for um we talk to domain experts we try and find the best people in the world to collaborate with um in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double\n\nin

In [81]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

GENERATION

In [82]:
answer = llm.invoke(final_prompt)
print(answer)

Yes, the topic of Nuclear fusion is discussed in the video. Specifically, it is mentioned as one of the areas where AI can help, alongside disease and biology. The discussion focuses on the challenges of building fusion reactors and contains plasma, and how AI can be applied to address the bottleneck problems that are currently stopping fusion from working.


Building a chain

In [83]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [84]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return context_text

In [85]:
parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
)

In [86]:
parallel_chain.invoke("who is denis?")

{'context': "demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time\n\nthe following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we 

In [87]:
parser = StrOutputParser()

In [88]:
main_chain = parallel_chain | prompt | llm | parser

In [89]:
main_chain.invoke("can u summarize the video in 2-3 lines?")

'The video discusses a method for designing and generating complex systems, including 3D structures, by allowing gradients and learning to flow from the output to the inputs. This approach, inspired by successes in games like AlphaGo and AlphaZero, aims to generalize across tasks and modalities, including language, to ultimately enable human-like understanding and operation in various domains.'